# 2026-10-04 被動量測資料品質核對

來源為使用者提供的兩份 CSV 及其指向的本機稽核紀錄。每列代表一個 URL 在同一批次的一次分析；兩個 CSV 是同一批次的不同輸出，不能相加為 100 個樣本。此 notebook 只讀本機檔案，不連線或執行網站內容。PhishHunt 是威脅情報來源，非人工真值。

In [1]:
import csv, json, hashlib, sys
from pathlib import Path
from collections import Counter
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").exists() and (p / "csv_reports").exists())
sys.path.insert(0, str(root))
names = ["batch_results_20261004_233606_196926.csv", "cloaking_report_20261004_233106_858340.csv"]
tables = [list(csv.DictReader((root / "csv_reports" / n).open(encoding="utf-8-sig", newline=""))) for n in names]
rows, report_rows = tables
key = lambda r: (r["run_id"], r["url"])
assert len({key(r) for r in rows}) == len(rows)
assert {key(r) for r in rows} == {key(r) for r in report_rows}
report_by_key = {key(r): r for r in report_rows}
shared = set(rows[0]) & set(report_rows[0])
projection_differences = Counter(c for r in rows for c in shared if r[c] != report_by_key[key(r)][c])
assert all(r[c] == report_by_key[key(r)][c] for r in rows for c in shared - {'is_phishing'})
audits = []
for r in rows:
    evidence = json.loads(r["evidence"])
    data = Path(evidence["audit_path"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == evidence["sha256"]
    audits.append(json.loads(data))
summary = {"sample_count": len(rows), "same_batch_projections": True, "duplicate_keys": 0, 'projection_differences': dict(projection_differences),
           "source_files_sha256": {n: hashlib.sha256((root / "csv_reports" / n).read_bytes()).hexdigest() for n in names},
           "audit_hashes_verified": len(audits),
           "observed_at_min": min(r["observed_at"] for r in rows), "observed_at_max": max(r["observed_at"] for r in rows),
           "verdicts": {f: dict(Counter(r[f] for r in rows)) for f in ("alive", "phishing_verdict", "cloaking_label", "source_label")}}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "sample_count": 50,
  "same_batch_projections": true,
  "duplicate_keys": 0,
  "projection_differences": {
    "is_phishing": 2
  },
  "source_files_sha256": {
    "batch_results_20261004_233606_196926.csv": "097caa0410a8fc26f6a33a2b5cb06336d6fec9dfec385bf7478d1ac6ef5a9f2f",
    "cloaking_report_20261004_233106_858340.csv": "60d8c08834feb3297e73f5b7fcc50f61d63e8851b2125a8941ad9db135b6e37c"
  },
  "audit_hashes_verified": 50,
  "observed_at_min": "2026-10-04T15:31:20.752906+00:00",
  "observed_at_max": "2026-10-04T15:36:06.181164+00:00",
  "verdicts": {
    "alive": {
      "": 2,
      "True": 48
    },
    "phishing_verdict": {
      "unknown": 27,
      "not_detected": 17,
      "phishing": 6
    },
    "cloaking_label": {
      "unknown": 50
    },
    "source_label": {
      "unlabeled": 50
    }
  }
}


In [2]:
unknown = [r for r in rows if r["phishing_verdict"] == "unknown"]
issues = Counter(x for r in unknown for x in json.loads(r["reason"])["phishing"].split(";"))
def cause_group(r):
    reasons = set(json.loads(r["reason"])["phishing"].split(";"))
    if any("UnicodeEncodeError" in x for x in reasons): return "pipeline_unicode_error"
    if "challenge_or_block_page" in reasons: return "challenge_or_block_page"
    if "http_unavailable" in reasons: return "http_unavailable"
    if "unreadable_content" in reasons: return "unreadable_content"
    if "content_truncated" in reasons: return "content_truncated"
    return "subresource_or_request_policy"
summary["unknown_reason_occurrences_overlapping"] = dict(issues)
summary["unknown_groups_exclusive_priority"] = dict(Counter(cause_group(r) for r in unknown))
summary["raw_rule_verdicts"] = dict(Counter(a.get("phishing_rule_result", {}).get("verdict", "not_run") for a in audits))
summary["rule_positive_but_final_unknown"] = sum(r["phishing_verdict"] == "unknown" and a.get("phishing_rule_result", {}).get("verdict") == "phishing" for r, a in zip(rows, audits))
summary["model_statuses"] = dict(Counter(m["status"] for a in audits for m in a.get("model_outputs", [])))
summary["ollama_missing_server_observations"] = sum("llama-server binary not found" in m.get("reason", "") for a in audits for m in a.get("model_outputs", []))
print(json.dumps({k: v for k, v in summary.items() if k.startswith(("unknown", "raw_rule", "rule_positive", "model_", "ollama_"))}, ensure_ascii=False, indent=2))

{
  "unknown_reason_occurrences_overlapping": {
    "UnicodeEncodeError: 'utf-8' codec can't encode characters in position 847-848: surrogates not allowed": 1,
    "UnicodeEncodeError: 'utf-8' codec can't encode characters in position 881-882: surrogates not allowed": 1,
    "content_truncated": 9,
    "request_failed": 16,
    "incomplete_due_to_request_policy": 9,
    "unreadable_content": 3,
    "challenge_or_block_page": 7,
    "http_unavailable": 2,
    "resource_http_error": 6
  },
  "unknown_groups_exclusive_priority": {
    "pipeline_unicode_error": 2,
    "content_truncated": 9,
    "unreadable_content": 3,
    "challenge_or_block_page": 7,
    "http_unavailable": 1,
    "subresource_or_request_policy": 5
  },
  "raw_rule_verdicts": {
    "not_run": 2,
    "not_detected": 32,
    "phishing": 16
  },
  "rule_positive_but_final_unknown": 10,
  "model_statuses": {
    "invalid_or_unavailable": 71
  },
  "ollama_missing_server_observations": 71
}


In [3]:
from fetch_feed import load_batch
pointer = json.loads((root / "active_batch.json").read_text(encoding="utf-8"))
batch = Path(pointer["path"])
manifest, source_records = load_batch(batch)
current_bytes, batch_bytes = (root / "urls.txt").read_bytes(), (batch / "urls.txt").read_bytes()
summary["provenance_check"] = {"batch_id": manifest["batch_id"], "snapshot_hashes_verified": True,
    "current_list_byte_equal": current_bytes == batch_bytes,
    "current_list_text_equal": (root / "urls.txt").read_text(encoding="utf-8") == (batch / "urls.txt").read_text(encoding="utf-8"),
    "root_crlf_count": current_bytes.count(b"\r\n"), "snapshot_crlf_count": batch_bytes.count(b"\r\n"),
    "csv_urls_match_snapshot": [r["url"] for r in rows] == list(source_records),
    "source_label_in_snapshot": manifest["source_label"], "source_fetched_at": manifest["fetched_at"]}
print(json.dumps(summary["provenance_check"], ensure_ascii=False, indent=2))
(root / "docs" / "experiment_20261004_review.json").write_text(json.dumps(summary, ensure_ascii=True, indent=2), encoding="utf-8")
print("Saved experiment_20261004_review.json")

{
  "batch_id": "d0791a9fd2bc47eaa23efb20d960d66c",
  "snapshot_hashes_verified": true,
  "current_list_byte_equal": false,
  "current_list_text_equal": true,
  "root_crlf_count": 50,
  "snapshot_crlf_count": 0,
  "csv_urls_match_snapshot": true,
  "source_label_in_snapshot": "phishunt_suspicious",
  "source_fetched_at": "2026-10-04T15:06:06.396716+00:00"
}
Saved experiment_20261004_review.json


判讀：釣魚 unknown 是觀察品質閘門與兩筆 Unicode 編碼錯誤的合計，不能當作陰性。原因出現次數可重疊；排他分組以 pipeline error → challenge → HTTP unavailable → unreadable → truncated → subresource/policy 的優先順序呈現。原始規則命中而 final unknown 仍不得回填為 phishing。cloaking 在真實被動相同 profile 下缺少受控對比，不能量測盛行率。兩份 CSV 皆來源未連接；快照核對可另作 provenance 補充，保留原始 CSV。沒有獨立真值，不能計算分類準確率、precision 或 recall。